npm GSM ARENA ENGINE

Information Retreival Lab
Harisaipravin Sv (17pw13)
Sathwik CH (17pw32)

Module 1 : web crawling

In [ ]:
import requests

data = requests.get("http://localhost:8888//gsmarena/brands")
data = data.json()
brand=[]
for i in data:
    url = "http://localhost:8888//gsmarena/brand/" + i['url']
    result = requests.get(url)
    brand.append(result.json())

print(brand)


In [ ]:
phone=[]
for i in brand:
    data = i['data']
    for j in data:
        url = "http://localhost:8888//gsmarena/phone/" + j['url']
        result = requests.get(url)
        phone.append(result.json()) 

Module 2 : Model Building

In [ ]:
#Extracting the crawlled data
import csv

filename = "gsm.csv"

fields = []
phone = []
with open(filename, 'r') as csvfile:
    csvreader = csv.reader(csvfile)
    fields = next(csvreader)
    for row in csvreader:
        phone.append(row)

    print("Total no. of rows: %d"%(csvreader.line_num))

print('Field names are: '+', '.join(field for field in fields))


Document preprocessing

In [ ]:
# Run if needed, doc is saved otherwise
import csv
from nltk.corpus import stopwords
def remove_punctuations(text):
    text=text.lower()
    punc = '''!()-[]{};:'"\,<>./?@#$%^&*_|~'''
    for i in text:
        if i in punc:
            text=text.replace(i,"")
    return text

documents=[]
for i in range(10000):
    temp=[]
    for j in phone[i]:
        if j=="":
            phone[i].remove("")
        elif j=="Yes":
            phone[i].remove("Yes")
        else:
            text = remove_punctuations(j)
            if text not in stopwords.words():
                temp.append(text)
    documents.append(temp)

#writting processed data to document
with open("out.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(documents)

In [ ]:
# extracting from existing document
import csv
datafile = open('out.csv', 'r')
datareader = csv.reader(datafile, delimiter=',')
documents = []
for row in datareader:
    documents.append(row)    

In [ ]:
#pre-processed text
print(documents)

Indexing List creation

In [ ]:
# Run only if needed 
# Generating pooling index table and saving it to file 
total_tokens=[]
for i in range(len(documents)):
    for j in documents[i]:
        if j not in total_tokens:
            total_tokens.append(j)
            
print("The total length of the terms is",len(total_tokens))

term_dict={}
for i in total_tokens:
    temp=[]
    for j in range(len(documents)):
        if i in documents[j]:
            temp.append(j+1)
    term_dict[i]=temp

f=open("phone_indexing","w")
for i in term_dict:
    string = ",".join(map(str,term_dict[i]))
    string = i + "," + string + "\n"
    f.write(string)

f.close()

In [ ]:
#Extracting from the already processed indexing list
import csv

data = {}
term_dict = {}
with open('phone_indexing') as fin:
    reader=csv.reader(fin, delimiter=",")
    for row in reader:
        data[row[0]]=row[1:]
        
for h in data:
    tt = []
    for j in data[h]:
        tt.append(int(j))
    term_dict[h] = tt

In [ ]:
# index dictionary
print(term_dict)

Tokenizing the words

In [ ]:
from nltk.tokenize import word_tokenize
from collections import Counter
import numpy as np
import math

processed_text = []
processed_title = []

for i in documents:
    processed_text.append(word_tokenize(str(i)))
    processed_title.append(word_tokenize(str((i[1]))))

In [ ]:
N = len(documents)
print(N)

df-idf model creation 

In [ ]:
#df building
DF = {}
for i in range(N):
    tokens = processed_text[i]
    for w in tokens:
        try:
            DF[w].add(i)
        except:
            DF[w] = {i}

    tokens = processed_title[i]
    for w in tokens:
        try:
            DF[w].add(i)
        except:
            DF[w] = {i}

In [ ]:
for i in DF:
    DF[i] = len(DF[i])
DF

In [ ]:
total_vocab_size = len(DF)
print(total_vocab_size)

In [ ]:
total_vocab = [x for x in DF]
print(total_vocab)

In [ ]:
def doc_freq(word):
    c = 0
    try:
        c = DF[word]
    except:
        pass
    return c

In [ ]:
#tf-idf calculation
doc = 0

tf_idf = {}

for i in range(N):
    
    tokens = processed_text[i]
    
    counter = Counter(tokens + processed_title[i])
    words_count = len(tokens + processed_title[i])
    
    for token in np.unique(tokens):
        
        tf = counter[token]/words_count
        df = doc_freq(token)
        idf = np.log((N+1)/(df+1))
        
        tf_idf[doc, token] = tf*idf

    doc += 1

In [ ]:
#Calculated tf-idf value
tf_idf

Module 3 : Estimating Model efficiency

In [ ]:
doc = 0

tf_idf_title = {}

for i in range(N):
    
    tokens = processed_title[i]
    counter = Counter(tokens + processed_text[i])
    words_count = len(tokens + processed_text[i])

    for token in np.unique(tokens):
        
        tf = counter[token]/words_count
        df = doc_freq(token)
        idf = np.log((N+1)/(df+1)) #numerator is added 1 to avoid negative values
        
        tf_idf_title[doc, token] = tf*idf

    doc += 1

Searching through entire Phone data

In [ ]:
print(tf_idf[(4, 'battery')])

Searching through Phone Titles

In [ ]:
print(tf_idf_title[(4, 'pixel')])

In [ ]:
#weightage
alpha = 0.3

In [ ]:
for i in tf_idf:
    tf_idf[i] *= alpha

In [ ]:
for i in tf_idf_title:
    tf_idf[i] = tf_idf_title[i]

In [ ]:
len(tf_idf)

Module 4 : Estimation and Ranking

In [ ]:
def matching_score(k, query):
    tokens = word_tokenize(str(query))

    print("Matching Score")
    print("\nQuery:", query)
    print("")
    print(tokens)
    
    query_weights = {}

    for key in tf_idf:
        
        if key[1] in tokens:
            try:
                query_weights[key[0]] += tf_idf[key]
            except:
                query_weights[key[0]] = tf_idf[key]
    
    query_weights = sorted(query_weights.items(), key=lambda x: x[1], reverse=True)

    print("")
    
    l = []
    
    for i in query_weights[:10]:
        l.append(i[0])
    
    for q in l:
        print(phone[q][0],phone[q][1])
    

In [ ]:
matching_score(10,'samsung s7')

Caluculaing similarity Score

In [ ]:
def cosine_sim(a, b):
    cos_sim = np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b))
    return cos_sim

In [ ]:
D = np.zeros((N, total_vocab_size))
for i in tf_idf:
    try:
        ind = total_vocab.index(i[1])
        D[i[0]][ind] = tf_idf[i]
    except:
        pass

In [ ]:
def gen_vector(tokens):

    Q = np.zeros((len(total_vocab)))
    
    counter = Counter(tokens)
    words_count = len(tokens)

    query_weights = {}
    
    for token in np.unique(tokens):
        
        tf = counter[token]/words_count
        df = doc_freq(token)
        idf = math.log((N+1)/(df+1))

        try:
            ind = total_vocab.index(token)
            Q[ind] = tf*idf
        except:
            pass
    return Q

In [ ]:
def cosine_similarity(k, query):
    print("Cosine Similarity")
    tokens = word_tokenize(str(query))
    
    print("\nQuery:", query)
    print("")
    print(tokens)
    
    d_cosines = []
    
    query_vector = gen_vector(tokens)
    
    for d in D:
        d_cosines.append(cosine_sim(query_vector, d))
        
    out = np.array(d_cosines).argsort()[-k:][::-1]
    
    print("")

    return out


In [ ]:
#Final Score Generated
cosine_similarity(10,'pixel dual camera')

In [ ]:
print(phone[3677])

Module 5 : Probablity model

In [ ]:
class BM25:

    def __init__(self, k1=1.5, b=0.75):
        self.b = b
        self.k1 = k1

    def fit(self, corpus):
        tf = []
        df = {}
        idf = {}
        doc_len = []
        corpus_size = 0
        for document in corpus:
            corpus_size += 1
            doc_len.append(len(document))

            # compute tf (term frequency) per document
            frequencies = {}
            for term in document:
                term_count = frequencies.get(term, 0) + 1
                frequencies[term] = term_count

            tf.append(frequencies)

            # compute df (document frequency) per term
            for term, _ in frequencies.items():
                df_count = df.get(term, 0) + 1
                df[term] = df_count

        for term, freq in df.items():
            idf[term] = math.log(1 + (corpus_size - freq + 0.5) / (freq + 0.5))

        self.tf_ = tf
        self.df_ = df
        self.idf_ = idf
        self.doc_len_ = doc_len
        self.corpus_ = corpus
        self.corpus_size_ = corpus_size
        self.avg_doc_len_ = sum(doc_len) / corpus_size
        return self

    def search(self, query):
        scores = [self._score(query, index) for index in range(self.corpus_size_)]
        return scores

    def _score(self, query, index):
        score = 0.0

        doc_len = self.doc_len_[index]
        frequencies = self.tf_[index]
        for term in query:
            if term not in frequencies:
                continue

            freq = frequencies[term]
            numerator = self.idf_[term] * freq * (self.k1 + 1)
            denominator = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avg_doc_len_)
            score += (numerator / denominator)

        return score

In [ ]:
def okapi(k,query):
    fileList = []
    fileList.append(filename)

    texts = documents

    query = [word for word in query.lower().split()]

    bm25 = BM25()
    bm25.fit(texts)
    scores = bm25.search(query)
    bm = []
    ot =[]
    for score, doc in zip(scores, documents):
        bm.append((doc, score))
        ot.append(score)

    ott = np.array(ot).argsort()[-k:][::-1]
    pd.DataFrame(bm, columns=['Document Title', 'Okapi Similarity'])

    return ott

In [ ]:
#okapi Calculation
import pandas as pd
fileList = []
fileList.append(filename)

#texts = [open(_file).read().split() for _file in fileList]
texts = documents

query = 'samsung s7 edge'
query = [word for word in query.lower().split()]

bm25 = BM25()
bm25.fit(texts)
scores = bm25.search(query)
bm = []
ot =[]
for score, doc in zip(scores, documents):
    bm.append((doc, score))
    ot.append(score)

ott = np.array(ot).argsort()[-10:][::-1]
pd.DataFrame(bm, columns=['Document Title', 'Okapi Similarity'])

In [ ]:
print(ott)

In [ ]:
#Final Okapi Output
for p in ott[:2]:
    print(phone[p][0],phone[p][1],bm[p])

Module 6 : Flask Creation for Frontend support

In [ ]:
import json
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route('/tf', methods=['POST'])
def index():
    query = request.args.get('query')
    print (query)
    output = cosine_similarity(10,query)
    print(output)
    kkk = []
    for h in output:
        if h<9999:
            op =[]
            op.append(phone[h][1])
            op.append(phone[h][0])
            kkk.append(op)
    print(kkk)
    return {
        "name" : kkk
    }

@app.route('/okapi', methods=['POST'])
def inde():
    c = 0
    query = request.args.get('query')
    print (query)
    output = okapi(10,query)
    print(output)
    kkk = []
    for h in output:
        c = c + 1
        if h<9999 and c != 1:
            op = []
            op.append(phone[h][1])
            op.append(phone[h][0])
            kkk.append(op)
    print(kkk)
    return {
        "name" : kkk
    }
app.run()

In [ ]:
# Query Checking
output = cosine_similarity(10,query)
print(output)
op =[]
for h in output:
    op.append(phone[h][1])
print(op)